# Use Monarch API
going from a CUI to MONDO


In [1]:
puts `cat ./maps/2025-biovista-disease.map`

biovista_umls,orphanet,snomed,name
C0027126,http://www.orpha.net/ORDO/Orphanet_273,http://purl.bioontology.org/ontology/SNOMEDCT/77956009,MYOTONIC DYSTROPHY TYPE 1
C0349653,http://www.orpha.net/ORDO/Orphanet_79318,http://purl.bioontology.org/ontology/SNOMEDCT/459063003,PMM2-CDG
C0023264,http://www.orpha.net/ORDO/Orphanet_506,http://purl.bioontology.org/ontology/SNOMEDCT/29570005,LEIGH SYNDROME
C0268467,http://www.orpha.net/ORDO/Orphanet_2102,http://purl.bioontology.org/ontology/SNOMEDCT/23447005,GTPCH DEFICIENCY
C0268631,http://www.orpha.net/ORDO/Orphanet_22,http://purl.bioontology.org/ontology/SNOMEDCT/49748000,SSADH DEFICIENCY
C0043459,http://www.orpha.net/ORDO/Orphanet_912,http://purl.bioontology.org/ontology/SNOMEDCT/88469006,ZELLWEGER SYNDROME
C0751882,http://www.orpha.net/ORDO/Orphanet_590,http://purl.bioontology.org/ontology/SNOMEDCT/230672006,CONGENITAL MYASTHENIC SYNDROME
C1849508,http://www.orpha.net/ORDO/Orphanet_3006,http://purl.bioontology.org/ontology/SNOMEDCT/733145009,P

In [2]:
#!/usr/bin/env ruby

require 'csv'
require 'rest-client'
require 'json'
require 'uri'
require 'net/http'

def get_mondo_label(mondo_uri)
  return nil unless mondo_uri && !mondo_uri.empty?
  local = mondo_uri.match(/MONDO_\d+/)&.[](0)
  return "" unless local
  iri     = "http://purl.obolibrary.org/obo/#{local}"
  encoded = URI.encode_uri_component(URI.encode_uri_component(iri))
  response = Net::HTTP.get_response(URI("https://www.ebi.ac.uk/ols4/api/ontologies/mondo/terms/#{encoded}"))
  return "" unless response.is_a?(Net::HTTPSuccess)
  JSON.parse(response.body)['label']
rescue => e
  warn "MONDO label lookup failed for #{mondo_uri}: #{e}"
  ""
end

# Configuration
INPUT_FILE    = './maps/2025-biovista-disease.map'
OUTPUT_FILE   = './maps/2026-biovista-disease-mondo.map'
NO_MATCH_FILE = './maps/2026-possible-phenotype-cuis.csv'

abort "Error: #{INPUT_FILE} not found!" unless File.exist?(INPUT_FILE)

# Read CSV and collect CUIs
rows = CSV.read(INPUT_FILE, headers: true)
cuis = rows.map { |row| row['biovista_umls'].strip }
puts cuis.size
abort "Error: No CUIs found in #{INPUT_FILE}" if cuis.empty?

warn "cuis #{cuis}"

# Build query parameters for Monarch API
entity_ids = cuis.map { |cui| "entity_id=UMLS:#{cui}" }.join('&')

puts "Querying Monarch API for #{cuis.size} CUIs..."
response = RestClient.get(
  "https://api-v3.monarchinitiative.org/v3/api/mappings?#{entity_ids}&format=json&limit=500&offset=0"
)
warn response.request.url

data = JSON.parse(response.body)
warn JSON.pretty_generate(response.body)

# Map CUI -> MONDO ID
mondo_map = {}
data['items'].each do |item|
  if item['object_id'] =~ /^UMLS:(C\d+)/
    mondo_map[$1] = item['subject_id']  # e.g. "MONDO:0018940"
  end
end

# Process rows
output_rows   = []
no_match_rows = []

rows.each do |row|
  cui = row['biovista_umls'].strip
  if (mondo_id = mondo_map[cui])
    mondo_uri      = "http://purl.obolibrary.org/obo/#{mondo_id.gsub(':', '_')}"
    row['mondo']   = mondo_uri
    canonical      = get_mondo_label(mondo_uri)
    row['name']    = canonical && !canonical.empty? ? canonical : row['name']
    warn "#{mondo_uri} => #{row['name']}"
    output_rows << row
  else
    no_match_rows << row
  end
end

# Write output CSV (with mondo column)
headers = rows.headers + ['mondo']
CSV.open(OUTPUT_FILE, 'w', write_headers: true, headers: headers) do |csv|
  output_rows.each { |row| csv << row }
end
puts "Wrote #{output_rows.size} rows with MONDO mappings to #{OUTPUT_FILE}"

# Write unmatched rows
if no_match_rows.any?
  CSV.open(NO_MATCH_FILE, 'w', write_headers: true, headers: rows.headers) do |csv|
    no_match_rows.each { |row| csv << row }
  end
  puts "Wrote #{no_match_rows.size} unmatched rows to #{NO_MATCH_FILE}"
else
  puts "All CUIs found a MONDO match!"
end

11


cuis ["C0027126", "C0349653", "C0023264", "C0268467", "C0268631", "C0043459", "C0751882", "C1849508", "C0024408", "C0268595", "C2931891"]


Querying Monarch API for 11 CUIs...


https://api-v3.monarchinitiative.org/v3/api/mappings?entity_id=UMLS:C0027126&entity_id=UMLS:C0349653&entity_id=UMLS:C0023264&entity_id=UMLS:C0268467&entity_id=UMLS:C0268631&entity_id=UMLS:C0043459&entity_id=UMLS:C0751882&entity_id=UMLS:C1849508&entity_id=UMLS:C0024408&entity_id=UMLS:C0268595&entity_id=UMLS:C2931891&format=json&limit=500&offset=0
"{\"limit\":500,\"offset\":0,\"total\":10,\"items\":[{\"subject_id\":\"MONDO:0010083\",\"subject_label\":\"succinic semialdehyde dehydrogenase deficiency\",\"predicate_id\":\"skos:exactMatch\",\"object_id\":\"UMLS:C0268631\",\"object_label\":null,\"mapping_justification\":\"semapv:UnspecifiedMatching\",\"id\":\"4fd6ee6c-4df3-462f-909c-5ee64e063264\",\"mapping_source\":[\"mondo.sssom\"],\"highlighting\":null},{\"subject_id\":\"MONDO:0009945\",\"subject_label\":\"pyridoxine-dependent epilepsy\",\"predicate_id\":\"skos:exactMatch\",\"object_id\":\"UMLS:C1849508\",\"object_label\":null,\"mapping_justification\":\"semapv:UnspecifiedMatching\",\"id\"

Wrote 10 rows with MONDO mappings to ./maps/2026-biovista-disease-mondo.map
Wrote 1 unmatched rows to ./maps/2026-possible-phenotype-cuis.csv


# Manual edits to source data — do not lose this

## Three CUIs not in original Biovista data

The following three rows were **manually added** to `maps/2025-biovista-disease.map` by Mark
because they appear in the Biovista edge file (`bv-kg-20250225.large`) but were absent from the
original Biovista disease list. Without them, those edges would silently be dropped during graphing.

They ARE now found by the Monarch API (confirmed — all three return MONDO mappings), so they will
be included automatically if the mapping cell is re-run against the source file. The key thing to
remember is that the source file itself was manually edited to add them.

| CUI      | Name in source file                                              | MONDO                    |
|----------|------------------------------------------------------------------|--------------------------|
| C1849508 | Pyridoxine-dependent developmental and epileptic encephalopathy  | MONDO:0009945            |
| C0024408 | Machado-Joseph disease                                           | MONDO:0007182            |
| C0268595 | Glutaryl-CoA dehydrogenase deficiency                            | MONDO:0009281            |

## C0023264 → C2931891 (UMLS synonym harmonization)

C0023264 (Leigh Syndrome) has a synonym CUI C2931891. The Monarch API maps only C2931891, not
C0023264. The original Biovista edge file uses C0023264, so a runtime substitution in the graphing
notebooks replaces C0023264 with C2931891 at lookup time. Both CUIs are in the source disease map
(C0023264 from the original data, C2931891 added manually for the Monarch mapping to work).

## Note on `# manually added` comments

The three manually added rows originally had `, # manually added` appended as a trailing CSV field.
This created a 5-column row against a 4-column header, pushing the `name` value out of position.
The comments have been stripped from all map files. Do not re-add inline CSV comments.